### Computing RDF2VEC for LUBM dataset

In [1]:
import io
import csv
import requests

res = requests.get(
    "http://127.0.0.1:8890/sparql/",
    params={
		"query": """
			SELECT DISTINCT ?id 
			FROM <lubm>
			WHERE { 
			{ ?id [] [] }
			UNION
			{ [] ?id [] }
			UNION
			{ [] [] ?id }
			}
		""",
		"format": "csv",
	},
)
str_io = io.StringIO(res.text)
reader = csv.DictReader(str_io)
entities = [row["id"] for row in reader]

In [2]:
import pandas as pd

from pyrdf2vec import RDF2VecTransformer
from pyrdf2vec.embedders import Word2Vec
from pyrdf2vec.graphs import KG
from pyrdf2vec.walkers import RandomWalker

In [ ]:
# %%prun -s cumulative

# Define our knowledge graph (here: DBPedia SPARQL endpoint).
knowledge_graph = KG(
    location="http://127.0.0.1:8890/sparql/",
    # location=r"..\datasets\lubm\graph\lubm.nt", # The in-memory version takes too much RAM and offers no visible speedup.
    skip_verify=True,
)

# Create our transformer, setting the embedding & walking strategy.
transformer = RDF2VecTransformer(
    embedder=Word2Vec(epochs=10),
    walkers=[RandomWalker(
        max_depth=4,
        max_walks=10,
        with_reverse=False,
        n_jobs=10,
    )],
    verbose=1
)

# Get our embeddings.
embeddings, literals = transformer.fit_transform(knowledge_graph, entities)

100%|██████████| 664048/664048 [1:03:33<00:00, 174.12it/s] 


Extracted 4277541 walks for 664048 entities (3814.1033s)
Fitted 4277541 walks (141.2895s)


In [ ]:
# Backing up the results into pickle file
import pickle

with open("../datasets/lubm/rdf2vec100dim.pkl", "wb") as f:
    pickle.dump(
        dict(zip(entities, embeddings)),
        f
    )

### Computing occurrences for LUBM dataset

In [27]:
import io
import csv
import requests

res = requests.get(
    "http://127.0.0.1:8890/sparql/",
    params={
		"query": """
			SELECT DISTINCT ?id, COUNT(?id) AS ?count
			FROM <lubm>
			WHERE { 
				{ ?id [] [] }
				UNION
				{ [] ?id [] }
				UNION
				{ [] [] ?id }
			}
		""",
		"format": "csv",
	},
)
str_io = io.StringIO(res.text)
reader = csv.DictReader(str_io)
count_vals = {row["id"]: int(row["count"]) for row in reader}

In [ ]:
import pickle

with open("../datasets/lubm/counts.pkl", "wb") as f:
	pickle.dump(count_vals, f)